In [6]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

In [ ]:
a = pd.read_csv("articles_data_updated.csv")
a[["PMC_ID", "Abstract"]].head()

,PMC_ID,Abstract
0,PMC4136787,We investigate the bioremediation potential of...
1,PMC3630201,Recent advances in the routine access to space...
2,PMC11988870,The oocytes of the African clawed frog (Xenopu...
3,PMC7998608,"Extra-intestinal pathogenicE. coli(ExPEC), inc..."
4,PMC5587110,Spaceflight poses risks to the central nervous...


In [19]:
# 1. Cargar datos
df = pd.read_csv("articles_data_updated.csv")
nulls_edad = df['Abstract'].isnull().sum()
print(f"Número de valores nulos en la columna 'Abstract': {nulls_edad}")

Número de valores nulos en la columna 'Abstract': 0


In [20]:
# IDs donde la columna NO es null
ids_con_nulos = df[df['Abstract'].isnull()]['PMC_ID'].tolist()
print(f"IDs con null: {ids_con_nulos}")

IDs con null: []


In [21]:
# Guardar en archivo .txt (uno por línea)
with open('ids_con_nulos.txt', 'w') as archivo:
    for id_valor in ids_con_nulos:
        archivo.write(f"{id_valor}\n")

print("Archivo 'ids_con_nulos.txt' guardado exitosamente")

Archivo 'ids_con_nulos.txt' guardado exitosamente


In [18]:
def completar_abstracts_simple(df):
    """
    Versión simple solo con URL e input manual
    """
    # Filtrar IDs sin abstract
    pendientes = df[df['Abstract'].isnull()]['PMC_ID'].tolist()
    
    print(f"🔍 {len(pendientes)} artículos sin abstract\n")
    
    for i, pmc_id in enumerate(pendientes, 1):
        print(f"\n{'='*50}")
        print(f"📄 Artículo {i} de {len(pendientes)}")
        print(f"🆔 PMC_ID: {pmc_id}")
        
        # Generar URL
        url = f"https://www.ncbi.nlm.nih.gov/pmc/articles/{pmc_id}/"
        print(f"🌐 {url}")
        
        # Input manual
        print("\n📝 Ingresa el abstract (presiona Enter 2 veces para finalizar):")
        print("   's' = Saltar, 'q' = Terminar")
        
        lineas = []
        while True:
            linea = input()
            if linea == '' and len(lineas) > 0:
                break
            elif linea.lower() == 's':
                print("⏭️  Saltado")
                break
            elif linea.lower() == 'q':
                print("🛑 Terminado por usuario")
                return df
            else:
                lineas.append(linea)
        
        if lineas and lineas[0].lower() not in ['s', 'q']:
            abstract = ' '.join(lineas).strip()
            df.loc[df['PMC_ID'] == pmc_id, 'Abstract'] = abstract
            print("✅ Abstract guardado")
    
    return df

# Uso rápido
df = pd.read_csv('articles_data_updated.csv')  # Cambia por tu archivo
df = completar_abstracts_simple(df)
df.to_csv('articles_data_updated.csv', index=False)
print("💾 Archivo guardado!")

🔍 53 artículos sin abstract


📄 Artículo 1 de 53
🆔 PMC_ID: PMC7072278
🌐 https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7072278/

📝 Ingresa el abstract (presiona Enter 2 veces para finalizar):
   's' = Saltar, 'q' = Terminar
✅ Abstract guardado

📄 Artículo 2 de 53
🆔 PMC_ID: PMC4378170
🌐 https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4378170/

📝 Ingresa el abstract (presiona Enter 2 veces para finalizar):
   's' = Saltar, 'q' = Terminar
✅ Abstract guardado

📄 Artículo 3 de 53
🆔 PMC_ID: PMC11167097
🌐 https://www.ncbi.nlm.nih.gov/pmc/articles/PMC11167097/

📝 Ingresa el abstract (presiona Enter 2 veces para finalizar):
   's' = Saltar, 'q' = Terminar
✅ Abstract guardado

📄 Artículo 4 de 53
🆔 PMC_ID: PMC6201722
🌐 https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6201722/

📝 Ingresa el abstract (presiona Enter 2 veces para finalizar):
   's' = Saltar, 'q' = Terminar
✅ Abstract guardado

📄 Artículo 5 de 53
🆔 PMC_ID: PMC5132293
🌐 https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5132293/

📝 Ingresa el abstra

In [ ]:
# 2. Modelo biomédico
model = SentenceTransformer('pritamdeka/S-PubMedBERT-MS-MARCO')

# 3. Generar embeddings (768 dimensiones)
vectors = np.vstack(df['Abstract'].apply(lambda x: model.encode(x)))

# 4. Reducir dimensión con PCA (por ejemplo, a 128 dimensiones)
pca = PCA(n_components=128)
vectors_reduced = pca.fit_transform(vectors)

# 5. Guardar los vectores reducidos
df['Vector'] = vectors_reduced.tolist()
df[['PMC_ID', 'Vector']].to_csv("abstracts_vectorized_pubmed_128d.csv", index=False)

print("✅ Guardado con éxito: 607 vectores de 128 dimensiones.")


✅ Guardado con éxito: 607 vectores de 128 dimensiones.


In [23]:
# Leer CSV y convertir a JSON
df = pd.read_csv('abstracts_vectorized_pubmed_128d.csv')
df.to_json('abstracts_vectorized_pubmed_128d.json', orient='records', indent=2)

print("✅ CSV convertido a JSON exitosamente")

✅ CSV convertido a JSON exitosamente
